# Lesson 07 - תבנית העיצוב של תכנון

מחברת זו מדגימה את **תבנית העיצוב של תכנון** לסוכנים של בינה מלאכותית באמצעות מסגרת הסוכנים של מייקרוסופט.
תלמד כיצד לפרק בקשות נסיעה מורכבות למשימות משנה מובנות, להקצות אותן לסוכנים מומחים,
ולתפעל את התוכנית הנוצרת — כל זאת באמצעות פלט מובנה בהשראת דגמי Pydantic.


## התקנה


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## פירוק משימה

פירוק משימה הוא הליבה של דפוס התכנון. במקום לבקש מסוכן יחיד לטפל בבקשה מורכבת מקצה לקצה, אנו מפצלים את הבעיה לתת-משימות קטנות ומוגדרות היטב. לכל תת-משימה מוקצה סוכן מומחה (לדוגמה, טיסות, בתי מלון, פעילויות) עם עדיפויות ברורות וסדר תלות.

שיטה זו מספקת כמה יתרונות:
- **בהירות**: לכל תת-משימה יש אחריות בודדת.
- **מקביליות**: תת-משימות בלתי תלויות יכולות לפעול במקביל.
- **אמינות**: כשלונות מבודדים לתת-משימות ספציפיות.
- **מעקב תקציב**: עלויות מוערכות לכל תת-משימה ומתועלות בסופו של דבר.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## יצירת סוכן תכנון עם פלט מובנה

סוכן התכנון פועל כ**מתאם דלפק קבלה**. בהתחשב בבקשת נסיעה ברמה גבוהה הוא מייצר `TravelPlan` מובנה — מפרק את הבקשה לתת-משימות, מגדיר עדיפויות ומזהה תלותיות כדי שמקבל שירות או שכבת ביצוע יוכלו לבצע את העבודה.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## ביצוע תוכנית עם כלים מומחים

ברגע שסוכן הקבלה יצר תוכנית מובנית, ה**סוכן הקונסיירז'** מבצע אותה.  
כל כלי מומחה מתמודד עם קטגוריה אחת של משימה משנית (טיסות, בתי מלון, פעילויות). הקונסיירז' עובר על המשימות המשניות בתוכנית לפי סדר התלות ומפנה כל אחת מהן לכלי המתאים.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## סיכום

בשיעור זה למדתם את **תבנית התכנון** לסוכני בינה מלאכותית:

1. **פירוק המשימה** — סוכן תכנון בקבלה מפרק בקשת נסיעה מורכבת למשימות משנה מובנות באמצעות מודלים של Pydantic, תוך הקצאת כל אחת לסוכן מומחה עם סדרי עדיפויות ותלויות.
2. **פלט מובנה** — על ידי העברת `response_format` הסוכן מחזיר אובייקט `TravelPlan` מאומת במקום טקסט חופשי, מה שהופך את העיבוד הבא לאמין.
3. **ביצוע התכנית** — סוכן קונסיירז' עובר על משימות המשנה באמצעות כלים מקצועיים (`book_flight`, `reserve_hotel`, `book_activity`) כדי לבצע את התכנית ולדווח על התוצאות.

תבנית זו מפרידה בין *מה לעשות* (תכנון) ל*איך לעשות* (ביצוע), מה שהופך את הסוכנים ליותר מודולריים, ניתנים לבחינה וקלים להרחבה.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב וויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון כי תרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. המסמך המקורי בשפתו המקורית יש להחשיב כמקור הסמכות. למידע חיוני, מומלץ להשתמש בתרגום מקצועי על ידי אדם. איננו אחראים לכל אי-הבנה או פרשנות מוטעית הנובעים משימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
